# 02 — Preprocessing : Dataset Multimodal Glaucome

Ce notebook crée :
- `GlaucomaPairedDataset` : Dataset PyTorch retournant des paires (Fundus, OCT, label)
- Transforms séparés Fundus / OCT avec data augmentation agressive
- StratifiedKFold(n_splits=5) pour validation croisée
- DataLoaders PyTorch prêts pour l'entraînement

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2

import torch
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold

print('✅ Imports OK')
print(f'   PyTorch        : {torch.__version__}')
print(f'   Albumentations : {A.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'   Device         : {DEVICE}')

In [ ]:
# ── Chemins & Config ───────────────────────────────────────────────────────
# IMPORTANT : chaque notebook Kaggle repart d'un environnement vierge.
# On reconstruit df_pairs ici directement (même logique que 01_eda.ipynb).

DATA_ROOT  = '/kaggle/input/datasets/marwanenasri/data-on-oct-and-fundus-images-zip/Data on OCT and Fundus Images'
EXCEL_PATH = os.path.join(DATA_ROOT, 'Patient Record-final.xlsx')
OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE      = 224
BATCH_SIZE    = 8
N_FOLDS       = 5
SEED          = 42
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Reconstruction df_pairs (copie de la logique EDA) ────────────────────
df_raw = pd.read_excel(EXCEL_PATH, header=1)
df_raw.columns = [str(c).strip() for c in df_raw.columns]

glaucoma_cols = [df_raw.columns[2], df_raw.columns[4], df_raw.columns[6], df_raw.columns[8]]
image_col     = df_raw.columns[0]

def majority_vote(row):
    votes = []
    for col in glaucoma_cols:
        val = str(row[col]).lower().strip()
        if val in ['yes', 'oui', '1']:    votes.append(1)
        elif val in ['no', 'non', '0']:   votes.append(0)
        elif 'suspect' in val:            votes.append(0.5)
    if not votes: return np.nan
    return 1 if np.mean(votes) >= 0.5 else 0

df_raw['label']      = df_raw.apply(majority_vote, axis=1)
df_raw['label_name'] = df_raw['label'].map({1: 'Glaucoma', 0: 'Normal'})
df_raw = df_raw.dropna(subset=['label']).reset_index(drop=True)

all_files = {}
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        all_files[f] = os.path.join(root, f)

records = []
for _, row in df_raw.iterrows():
    raw_name = str(row[image_col]).strip().strip("'\"*")
    if not raw_name.lower().endswith('.jpg'):
        raw_name += '.jpg'
    oct_name    = raw_name.replace('Color', 'B-scan')
    fundus_path = all_files.get(raw_name)
    oct_path    = all_files.get(oct_name)
    if fundus_path and oct_path:
        records.append({
            'fundus_path': fundus_path,
            'oct_path'   : oct_path,
            'label'      : int(row['label']),
            'label_name' : row['label_name'],
        })

df_pairs = pd.DataFrame(records)
print(f'✅ Dataset reconstruit : {len(df_pairs)} paires')
print(df_pairs['label_name'].value_counts())
df_pairs.head(3)

In [ ]:
# ── Transforms Albumentations (API >= 1.4) ─────────────────────────────────
# Changements nouvelle API :
#   ShiftScaleRotate       → A.Affine
#   GaussNoise(var_limit)  → GaussNoise(std_range)
#   CoarseDropout(max_holes, max_height, max_width, fill_value)
#                          → CoarseDropout(num_holes_range, hole_height_range, hole_width_range, fill)

def get_fundus_train_transform(img_size=IMG_SIZE):
    """Augmentation agressive pour Fundus (RGB)."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.Affine(
            translate_percent={'x': (-0.1, 0.1), 'y': (-0.1, 0.1)},
            scale=(0.85, 1.15),
            rotate=(-30, 30),
            mode=cv2.BORDER_REFLECT,
            p=0.7
        ),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.4),
        A.CLAHE(clip_limit=2.0, p=0.3),
        A.GaussNoise(std_range=(0.03, 0.15), p=0.4),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5)),
            A.MotionBlur(blur_limit=5),
            A.MedianBlur(blur_limit=3),
        ], p=0.3),
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(8, 20),
            hole_width_range=(8, 20),
            fill=0,
            p=0.3
        ),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def get_fundus_val_transform(img_size=IMG_SIZE):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def get_oct_train_transform(img_size=IMG_SIZE):
    """Augmentation agressive pour OCT (grayscale → traité en RGB)."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent={'x': (-0.08, 0.08), 'y': (-0.08, 0.08)},
            scale=(0.90, 1.10),
            rotate=(-10, 10),
            mode=cv2.BORDER_REFLECT,
            p=0.6
        ),
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.35, p=0.6),
        A.CLAHE(clip_limit=3.0, p=0.4),
        A.GaussNoise(std_range=(0.02, 0.10), p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5)),
            A.MedianBlur(blur_limit=3),
        ], p=0.3),
        A.CoarseDropout(
            num_holes_range=(1, 6),
            hole_height_range=(5, 15),
            hole_width_range=(10, 40),
            fill=0,
            p=0.25
        ),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def get_oct_val_transform(img_size=IMG_SIZE):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


print(f'✅ Albumentations {A.__version__} — Transforms définis')
print(f'   Fundus train : {len(get_fundus_train_transform().transforms)} transforms')
print(f'   OCT train    : {len(get_oct_train_transform().transforms)} transforms')

In [ ]:
# ── GlaucomaPairedDataset ──────────────────────────────────────────────────
class GlaucomaPairedDataset(Dataset):
    """
    Dataset multimodal retournant des paires (Fundus, OCT, label).

    Retourne un dict :
        {
            'fundus' : FloatTensor [3, H, W],
            'oct'    : FloatTensor [3, H, W],
            'label'  : LongTensor  scalar,
            'index'  : int
        }
    """

    def __init__(self, df, fundus_transform, oct_transform, mode='train'):
        self.df               = df.reset_index(drop=True)
        self.fundus_transform = fundus_transform
        self.oct_transform    = oct_transform
        self.mode             = mode

    def __len__(self):
        return len(self.df)

    def _load_fundus(self, path):
        """Charge une image Fundus en RGB numpy uint8."""
        return np.array(Image.open(path).convert('RGB'), dtype=np.uint8)

    def _load_oct(self, path):
        """Charge une image OCT : grayscale → RGB numpy uint8."""
        return np.array(Image.open(path).convert('L').convert('RGB'), dtype=np.uint8)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        fundus_np = self._load_fundus(row['fundus_path'])
        oct_np    = self._load_oct(row['oct_path'])

        fundus_tensor = self.fundus_transform(image=fundus_np)['image']
        oct_tensor    = self.oct_transform(image=oct_np)['image']

        label = torch.tensor(int(row['label']), dtype=torch.long)

        return {
            'fundus' : fundus_tensor,
            'oct'    : oct_tensor,
            'label'  : label,
            'index'  : idx
        }


# ── Test rapide ─────────────────────────────────────────────────────────────
_ds_test = GlaucomaPairedDataset(
    df=df_pairs,
    fundus_transform=get_fundus_train_transform(),
    oct_transform=get_oct_train_transform(),
    mode='train'
)
sample = _ds_test[0]

print('✅ GlaucomaPairedDataset OK')
print(f'   Fundus shape : {sample["fundus"].shape}   dtype={sample["fundus"].dtype}')
print(f'   OCT shape    : {sample["oct"].shape}   dtype={sample["oct"].dtype}')
print(f'   Label        : {sample["label"]}')
print(f'   Dataset len  : {len(_ds_test)}')

In [ ]:
# ── StratifiedKFold ────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

X = df_pairs.index.values
y = df_pairs['label'].values

folds = list(skf.split(X, y))

print(f'✅ StratifiedKFold : {N_FOLDS} folds')
print(f'{"Fold":<6} {"Train":<8} {"Val":<6} {"Train G/N":<14} {"Val G/N"}')
print('-' * 50)
for i, (train_idx, val_idx) in enumerate(folds):
    y_tr = y[train_idx]
    y_va = y[val_idx]
    print(f'{i+1:<6} {len(train_idx):<8} {len(val_idx):<6} '
          f'{y_tr.sum()}/{len(y_tr)-y_tr.sum():<8} '
          f'{y_va.sum()}/{len(y_va)-y_va.sum()}')

In [ ]:
# ── create_fold_loaders ────────────────────────────────────────────────────
def create_fold_loaders(df, train_idx, val_idx,
                        batch_size=BATCH_SIZE,
                        img_size=IMG_SIZE,
                        num_workers=2):
    """
    Crée les DataLoaders train/val pour un fold donné.
    Réutilisable dans les notebooks 03, 04, 05.
    """
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_val   = df.iloc[val_idx].reset_index(drop=True)

    ds_train = GlaucomaPairedDataset(
        df=df_train,
        fundus_transform=get_fundus_train_transform(img_size),
        oct_transform=get_oct_train_transform(img_size),
        mode='train'
    )
    ds_val = GlaucomaPairedDataset(
        df=df_val,
        fundus_transform=get_fundus_val_transform(img_size),
        oct_transform=get_oct_val_transform(img_size),
        mode='val'
    )

    train_loader = DataLoader(
        ds_train, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=(DEVICE == 'cuda'), drop_last=False
    )
    val_loader = DataLoader(
        ds_val, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=(DEVICE == 'cuda'), drop_last=False
    )
    return train_loader, val_loader


# ── Test fold 0 ──────────────────────────────────────────────────────────────
train_idx_0, val_idx_0 = folds[0]
train_loader_0, val_loader_0 = create_fold_loaders(df_pairs, train_idx_0, val_idx_0)

batch = next(iter(train_loader_0))
print('✅ DataLoaders OK (fold 0)')
print(f'   Fundus batch : {batch["fundus"].shape}')
print(f'   OCT batch    : {batch["oct"].shape}')
print(f'   Labels       : {batch["label"].tolist()}')
print(f'   Train batches: {len(train_loader_0)}')
print(f'   Val batches  : {len(val_loader_0)}')

In [ ]:
# ── Visualisation augmentations ────────────────────────────────────────────
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).permute(1, 2, 0).numpy().clip(0, 1)

N_AUG      = 5
sample_row = df_pairs.iloc[0]
fundus_np  = np.array(Image.open(sample_row['fundus_path']).convert('RGB'), dtype=np.uint8)
oct_np     = np.array(Image.open(sample_row['oct_path']).convert('L').convert('RGB'), dtype=np.uint8)

fig, axes = plt.subplots(2, N_AUG + 1, figsize=((N_AUG + 1) * 3, 6))
fig.suptitle('Augmentations — Fundus (haut) / OCT (bas)', fontsize=13, fontweight='bold')

orig_f = np.array(Image.open(sample_row['fundus_path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE)))
orig_o = np.array(Image.open(sample_row['oct_path']).convert('L').convert('RGB').resize((IMG_SIZE, IMG_SIZE)))
axes[0, 0].imshow(orig_f);  axes[0, 0].set_title('Original', fontweight='bold', fontsize=9); axes[0, 0].axis('off')
axes[1, 0].imshow(orig_o);  axes[1, 0].set_title('Original', fontweight='bold', fontsize=9); axes[1, 0].axis('off')

f_tfm = get_fundus_train_transform()
o_tfm = get_oct_train_transform()
for j in range(N_AUG):
    axes[0, j+1].imshow(denormalize(f_tfm(image=fundus_np)['image']))
    axes[0, j+1].set_title(f'Aug {j+1}', fontsize=9); axes[0, j+1].axis('off')
    axes[1, j+1].imshow(denormalize(o_tfm(image=oct_np)['image']))
    axes[1, j+1].set_title(f'Aug {j+1}', fontsize=9); axes[1, j+1].axis('off')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/augmentations_viz.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Visualisation augmentations sauvegardée')

In [ ]:
# ── Visualisation distribution folds ──────────────────────────────────────
fig, axes = plt.subplots(1, N_FOLDS, figsize=(N_FOLDS * 3, 4))
fig.suptitle('Distribution classes par fold', fontsize=13, fontweight='bold')

for i, (train_idx, val_idx) in enumerate(folds):
    y_tr = y[train_idx]
    y_va = y[val_idx]
    cats   = ['Tr\nNorm', 'Tr\nGlauc', 'Va\nNorm', 'Va\nGlauc']
    vals   = [(y_tr==0).sum(), (y_tr==1).sum(), (y_va==0).sum(), (y_va==1).sum()]
    colors = ['#FF8080', '#C00000', '#70B0E0', '#1F4E79']
    bars   = axes[i].bar(cats, vals, color=colors, edgecolor='white', linewidth=1.2)
    for bar, v in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, v + 0.1,
                     str(v), ha='center', fontsize=9, fontweight='bold')
    axes[i].set_title(f'Fold {i+1}', fontweight='bold')
    axes[i].set_ylim(0, max(vals) + 4)
    axes[i].tick_params(axis='x', labelsize=7)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/folds_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Distribution folds sauvegardée')

In [ ]:
# ── Sauvegarde config JSON ─────────────────────────────────────────────────
import json

config = {
    'img_size'      : IMG_SIZE,
    'batch_size'    : BATCH_SIZE,
    'n_folds'       : N_FOLDS,
    'seed'          : SEED,
    'n_pairs'       : len(df_pairs),
    'n_glaucoma'    : int((df_pairs['label'] == 1).sum()),
    'n_normal'      : int((df_pairs['label'] == 0).sum()),
    'imagenet_mean' : IMAGENET_MEAN,
    'imagenet_std'  : IMAGENET_STD,
    'device'        : DEVICE
}

with open(f'{OUTPUT_DIR}/preprocessing_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('\n✅ Preprocessing terminé !')
print('─' * 40)
for k, v in config.items():
    print(f'   {k:<20} : {v}')
print('\n➡️  Prochain notebook : 03_fundus_model.ipynb')